# 📖 Hướng Dẫn Đọc Dữ Liệu — Ames Housing Dataset

> **Mục đích của notebook này:** Giúp bạn hiểu rõ ý nghĩa của từng cột trong dataset trước khi bắt tay vào phân tích và xây dựng model.

Dataset Ames Housing gồm **81 cột** mô tả các đặc điểm của ngôi nhà được bán tại thành phố Ames, Iowa (Mỹ) từ năm 2006–2010. Mục tiêu cuối cùng là dự đoán **SalePrice** (giá bán).

---

## 🗂️ Phân Loại Các Cột Theo Nhóm

| Nhóm | Số cột | Chủ đề |
|---|---|---|
| 1 | 11 | Vị trí & Lô đất |
| 2 | 6 | Loại nhà & Phong cách xây dựng |
| 3 | 7 | Chất lượng tổng thể & Thời gian |
| 4 | 11 | Tầng hầm (Basement) |
| 5 | 14 | Diện tích sàn & Phòng bên trong |
| 6 | 9 | Garage |
| 7 | 9 | Bên ngoài nhà (Mái, Tường, Hiên, Bể bơi) |
| 8 | 6 | Thông tin giao dịch bán |
| — | 1 | **SalePrice** — Target cần dự đoán |

---

## 🔖 Ký Hiệu Dùng Trong Notebook

| Ký hiệu | Nghĩa |
|---|---|
| 🔢 Numeric | Cột số — dùng trực tiếp |
| 📊 Ordinal | Cột có thứ tự (Ex > Gd > TA > Fa > Po) → cần **Ordinal Encoding** |
| 🏷️ Nominal | Cột phân loại không có thứ tự → cần **One-Hot Encoding** |
| 🎯 Target | Biến mục tiêu cần dự đoán |

| Bước pipeline | Viết tắt |
|---|---|
| Buổi 3 — EDA | `[EDA]` |
| Buổi 4 — Data Cleaning & Preprocessing | `[B4]` |
| Buổi 5 — Feature Engineering | `[B5]` |
| Buổi 6 — Model Training | `[B6]` |

---

## 🗺️ Nhóm 1 — Vị Trí & Lô Đất (11 cột)

> Nhóm này mô tả **lô đất** (mảnh đất ngôi nhà đứng trên) và **vị trí địa lý** của nó. Rất quan trọng vì bất động sản = "location, location, location".

---

### `MSZoning` — Phân vùng quy hoạch 🏷️ Nominal

Khu vực được **chính quyền quy hoạch** cho mục đích sử dụng gì.

| Giá trị | Nghĩa | Ảnh hưởng đến giá |
|---|---|---|
| `RL` | Khu dân cư mật độ thấp (nhà rộng, vườn lớn) | Cao ↑ |
| `RM` | Khu dân cư mật độ trung bình (chung cư, townhouse) | Trung bình |
| `FV` | Khu dân cư làng nổi | Cao ↑ |
| `C` | Khu thương mại | Thấp ↓ (ít khi bán nhà ở đây) |
| `A` | Khu nông nghiệp | Thấp ↓ |

> 🎯 **Dùng ở:** `[EDA]` phân tích phân phối, `[B4]` One-Hot Encoding, `[B6]` feature quan trọng của tree model

---

### `LotFrontage` — Chiều rộng mặt tiền 🔢 Numeric

Số **feet tuyến tính** của mảnh đất tiếp giáp đường phố.

- Nhà có mặt tiền rộng → dễ tiếp cận → giá cao hơn
- **Missing ~17%** → đây là cột cần xử lý đặc biệt ở Buổi 4!
- Chiến lược: điền median theo từng `Neighborhood` (khu phố cùng loại có mặt tiền tương tự)

> 🎯 **Dùng ở:** `[B4]` fill missing theo group, `[B5]` có thể tạo feature "LotFrontage / LotArea" = tỷ lệ mặt tiền

---

### `LotArea` — Diện tích lô đất 🔢 Numeric

Tổng diện tích mảnh đất (square feet). **1 sq ft ≈ 0.093 m²**

- Trung bình ~10,500 sqft ≈ 975 m²
- Phân phối rất lệch phải (skewed right) — có vài lô đất cực lớn
- Cần **log transform** để model hoạt động tốt hơn

> 🎯 **Dùng ở:** `[EDA]` kiểm tra skew, `[B5]` log transform nếu cần, `[B6]` feature quan trọng

---

### `Street` — Loại đường tiếp cận 🏷️ Nominal

Loại đường dẫn vào nhà: `Pave` (đường nhựa) hoặc `Grvl` (đường đá dăm).

- 99.6% là `Pave` → gần như không có variation → **ít giá trị dự đoán**
- Có thể drop hoặc giữ lại — model sẽ tự bỏ qua

> 🎯 **Dùng ở:** `[EDA]` phát hiện cột low-variance, `[B4]` One-Hot

---

### `Alley` — Loại đường hẻm phía sau 🏷️ Nominal

Kiểu đường hẻm dẫn vào phía sau nhà: `Pave`, `Grvl`, hoặc `NA` (không có hẻm).

- **Missing ~93%** — nhưng NA có nghĩa là "không có hẻm", KHÔNG phải thiếu dữ liệu!
- Điền `"None"` thay vì xóa

> 🎯 **Dùng ở:** `[B4]` điền "None" cho missing, `[B4]` One-Hot

---

### `LotShape` — Hình dạng lô đất 📊 Ordinal

Mức độ đều đặn của hình dạng mảnh đất.

| Giá trị | Nghĩa |
|---|---|
| `Reg` | Hình chữ nhật đều (Regular) |
| `IR1` | Hơi không đều (Slightly Irregular) |
| `IR2` | Khá không đều (Moderately Irregular) |
| `IR3` | Rất không đều (Irregular) |

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding (Reg=3, IR1=2, IR2=1, IR3=0)

---

### `LandContour` — Độ bằng phẳng 🏷️ Nominal

Địa hình của lô đất nhìn tổng thể.

| Giá trị | Nghĩa |
|---|---|
| `Lvl` | Phẳng gần như hoàn toàn (tốt nhất) |
| `Bnk` | Lô đất dốc lên cao từ đường |
| `HLS` | Dốc nghiêng từ cạnh này sang cạnh kia |
| `Low` | Lô đất thấp hơn mặt đường (dễ ngập) |

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

### `Utilities` — Tiện ích công cộng 🏷️ Nominal

Loại tiện ích (điện, nước, ga, cống) được kết nối.

- Gần 100% là `AllPub` (đầy đủ tiện ích) → **cột này gần như vô dụng**
- Thường bị drop trong các pipeline thực tế

> 🎯 **Dùng ở:** `[EDA]` phát hiện low-variance, thường **bị drop**

---

### `LotConfig` — Cấu hình lô đất 🏷️ Nominal

Vị trí của lô đất trong khu dân cư.

| Giá trị | Nghĩa |
|---|---|
| `Inside` | Lô đất bên trong (không góc đường) |
| `Corner` | Lô đất góc đường (2 mặt tiếp đường) |
| `CulDSac` | Cuối đường cụt (yên tĩnh, riêng tư) |
| `FR2` | Tiếp giáp 2 mặt đường |
| `FR3` | Tiếp giáp 3 mặt đường |

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

### `LandSlope` — Độ dốc lô đất 📊 Ordinal

Mức độ dốc của mảnh đất: `Gtl` (nhẹ) → `Mod` (trung bình) → `Sev` (dốc nhiều).

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding (Gtl=2, Mod=1, Sev=0)

---

### `Neighborhood` — Khu phố 🏷️ Nominal ⭐ Quan trọng nhất nhóm này

Khu phố vật lý trong thành phố Ames — 25 khu phố khác nhau.

- **Feature quan trọng nhất** trong nhóm Location: khu phố = giá nhà
- Cardinality cao (25) → One-Hot tạo 24 cột mới
- Nâng cao: có thể dùng **Target Encoding** để nén về 1 cột

> 🎯 **Dùng ở:** `[EDA]` boxplot giá theo khu phố, `[B4]` One-Hot / Target Encoding, `[B5]` có thể tạo "NeighborhoodTier" (phân nhóm khu phố theo giá)

---

### `Condition1` & `Condition2` — Điều kiện lân cận 🏷️ Nominal

Nhà gần những gì đặc biệt: đường lớn, đường sắt, công viên, v.v.

| Giá trị | Nghĩa | Ảnh hưởng |
|---|---|---|
| `Norm` | Bình thường | Trung lập |
| `PosN`, `PosA` | Gần công viên / khu xanh | Tốt ↑ |
| `Artery`, `Feedr` | Gần đường lớn / ồn | Xấu ↓ |
| `RRAn`, `RRAe` | Sát đường sắt | Xấu ↓ |

- `Condition2` chỉ dùng khi có từ 2 điều kiện đặc biệt trở lên

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

## 🏠 Nhóm 2 — Loại Nhà & Phong Cách Xây Dựng (6 cột)

> Nhóm này mô tả **loại căn nhà** (nhà đơn, duplex, townhouse...) và **kiểu dáng kiến trúc**. Ảnh hưởng trực tiếp đến diện tích sử dụng và giá trị.

---

### `MSSubClass` — Mã loại nhà 🏷️ Nominal ⚠️ Dễ nhầm!

Mã số xác định **kiểu xây dựng** của căn nhà.

> ⚠️ **Cạm bẫy phổ biến:** Cột này trông như số nguyên (20, 30, 60...) nhưng **KHÔNG phải số liên tục**! 20 không có nghĩa là "tốt hơn" 10. Đây là mã danh mục!

| Giá trị | Nghĩa |
|---|---|
| `20` | Nhà 1 tầng xây từ 1946 trở về sau |
| `60` | Nhà 2 tầng xây từ 1946 trở về sau |
| `120` | Nhà 1 tầng kiểu PUD (khu đô thị có quy hoạch) |
| `160` | Nhà 2 tầng kiểu PUD |
| `90` | Duplex (2 hộ gia đình) |
| `30` | Nhà 1 tầng xây trước 1945 (cũ) |

> 🎯 **Dùng ở:** `[B4]` phải convert sang string trước khi One-Hot Encoding (vì dtype là int nhưng là nominal!)

---

### `BldgType` — Loại công trình 🏷️ Nominal

Phân loại **cấu trúc** của căn nhà.

| Giá trị | Nghĩa | Ảnh hưởng giá |
|---|---|---|
| `1Fam` | Nhà đơn gia đình (phổ biến nhất ~83%) | Cao nhất ↑ |
| `TwnhsE` | Townhouse — căn góc (End Unit) | Khá cao |
| `TwnhsI` | Townhouse — căn giữa (Inside Unit) | Trung bình |
| `Duplx` | Duplex — 2 hộ gia đình | Trung bình |
| `2FmCon` | Nhà đơn đã chuyển đổi thành 2 hộ | Thấp |

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

### `HouseStyle` — Phong cách kiến trúc 🏷️ Nominal

Số tầng và kiểu thiết kế của ngôi nhà.

| Giá trị | Nghĩa |
|---|---|
| `1Story` | Nhà 1 tầng |
| `2Story` | Nhà 2 tầng |
| `1.5Fin` | Nhà 1.5 tầng — tầng áp mái hoàn thiện |
| `1.5Unf` | Nhà 1.5 tầng — tầng áp mái chưa hoàn thiện |
| `SFoyer` | Split Foyer (tiền sảnh chia tầng) |
| `SLvl` | Split Level (nhà chia cấp) |

- Liên quan trực tiếp đến `2ndFlrSF` (diện tích tầng 2)
- Nhà `2Story` thường có `2ndFlrSF` > 0, nhà `1Story` thì không

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding, `[B5]` cross-check với `2ndFlrSF`

---

### `RoofStyle` — Kiểu mái 🏷️ Nominal

Hình dạng của mái nhà: `Gable` (mái đôi dốc — phổ biến nhất), `Hip` (mái 4 mặt dốc), `Flat` (mái bằng), `Gambrel`, `Mansard`, `Shed`.

- Ít ảnh hưởng đến giá trong dataset này
- Nhưng là feature hữu ích cho model biết về "phong cách xây dựng"

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

### `RoofMatl` — Vật liệu mái 🏷️ Nominal

Chất liệu làm mái: `CompShg` (ngói composite — chiếm ~98%), `WdShngl`, `Metal`, `Tar&Grv`...

- Gần như chỉ có 1 giá trị → **low-variance**, ít có ý nghĩa dự đoán
- Trong các pipeline chuẩn thường bị drop

> 🎯 **Dùng ở:** `[EDA]` kiểm tra phân phối, thường **bị drop hoặc giữ lại không đáng kể**

---

### `Foundation` — Loại móng nhà 🏷️ Nominal

Chất liệu và kiểu móng của ngôi nhà.

| Giá trị | Nghĩa | Ghi chú |
|---|---|---|
| `PConc` | Móng bê tông đổ tại chỗ | Hiện đại, chắc chắn nhất |
| `CBlock` | Khối xi măng | Phổ biến nhà cũ |
| `BrkTil` | Gạch & ngói | Cũ, ít phổ biến |
| `Slab` | Nền bê tông (không tầng hầm) | Ít gặp ở Ames |
| `Stone` | Đá | Rất cũ |
| `Wood` | Gỗ | Hiếm |

- Nhà `PConc` thường mới hơn và đắt hơn
- Liên quan chặt với `YearBuilt` và `BsmtQual`

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

## ⭐ Nhóm 3 — Chất Lượng Tổng Thể & Thời Gian (7 cột)

> Nhóm này chứa **những feature quan trọng nhất** của toàn bộ dataset. `OverallQual` một mình giải thích được ~65% variance của SalePrice!

---

### `OverallQual` — Chất lượng tổng thể 🔢 Numeric (thực ra là Ordinal) ⭐⭐⭐

**Đánh giá chất lượng vật liệu xây dựng và hoàn thiện** của ngôi nhà, từ 1 đến 10.

| Điểm | Nhận xét |
|---|---|
| 10 | Very Excellent (sang trọng tuyệt hảo) |
| 9 | Excellent |
| 8 | Very Good |
| 7 | Good |
| 6 | Above Average |
| 5 | Average (trung bình — phổ biến nhất) |
| 4 | Below Average |
| 3 | Fair |
| 2 | Poor |
| 1 | Very Poor |

- **Tương quan cao nhất với SalePrice** trong toàn bộ dataset (r ≈ 0.79)
- Nhà `OverallQual = 10` có thể đắt gấp 5-6 lần nhà `OverallQual = 1`
- Có thể dùng **trực tiếp như số** vì đã có thứ tự ý nghĩa

> 🎯 **Dùng ở:** `[EDA]` boxplot giá theo OverallQual, `[B6]` feature quan trọng nhất, `[B5]` tạo feature tương tác với `GrLivArea`

---

### `OverallCond` — Tình trạng tổng thể 🔢 Numeric (Ordinal)

**Đánh giá tình trạng hiện tại** (condition) của ngôi nhà, thang 1–10.

- Khác với `OverallQual`: Qual = chất lượng vật liệu khi xây, Cond = tình trạng bảo dưỡng hiện tại
- Một ngôi nhà cũ (`OverallQual` thấp) nhưng được chăm sóc tốt vẫn có `OverallCond` cao
- Tương quan với SalePrice thấp hơn nhiều so với `OverallQual` (r ≈ 0.10)

> 🎯 **Dùng ở:** `[EDA]` so sánh phân phối với OverallQual, `[B5]` tạo "QualCondScore = OverallQual * OverallCond"

---

### `YearBuilt` — Năm xây dựng 🔢 Numeric ⭐⭐

Năm ngôi nhà được xây dựng lần đầu (1872–2010 trong dataset này).

- Nhà mới hơn → giá cao hơn (correlation r ≈ 0.52)
- **Không nên dùng trực tiếp** — thay vào đó tính **"tuổi nhà"**
- `HouseAge = YrSold - YearBuilt` → feature trực quan hơn cho model

> 🎯 **Dùng ở:** `[EDA]` histogram phân phối năm xây, `[B5]` → tạo `HouseAge`, `RemodAge`

---

### `YearRemodAdd` — Năm sửa chữa gần nhất 🔢 Numeric

Năm sửa chữa/nâng cấp gần nhất. Nếu không sửa bao giờ, giá trị = `YearBuilt`.

- Nhà vừa được sửa → giá cao hơn
- `RemodAge = YrSold - YearRemodAdd` → thời gian kể từ lần sửa gần nhất
- `IsRemodeled = 1 if YearRemodAdd > YearBuilt else 0` → có được sửa không?

> 🎯 **Dùng ở:** `[B5]` tạo `RemodAge`, `IsRemodeled`

---

### `ExterQual` — Chất lượng vật liệu ngoại thất 📊 Ordinal ⭐⭐

Đánh giá chất lượng vật liệu bên ngoài nhà.

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `Ex` | 4 | Excellent |
| `Gd` | 3 | Good |
| `TA` | 2 | Typical/Average (phổ biến nhất) |
| `Fa` | 1 | Fair |
| `Po` | 0 | Poor (gần như không xuất hiện) |

- Tương quan khá cao với SalePrice (r ≈ 0.68)
- Cùng thang đo với `BsmtQual`, `KitchenQual`, `GarageQual`...

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding (Ex=4, Gd=3, TA=2, Fa=1, Po=0)

---

### `ExterCond` — Tình trạng hiện tại ngoại thất 📊 Ordinal

Đánh giá **tình trạng hiện tại** (không phải chất lượng gốc) của vật liệu bên ngoài.

- Cùng thang điểm Ex/Gd/TA/Fa/Po
- Phần lớn nhà là `TA` → ít variance hơn `ExterQual`
- Tương quan thấp hơn với giá

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding

---

### `KitchenQual` — Chất lượng nhà bếp 📊 Ordinal ⭐⭐

Đánh giá chất lượng nhà bếp (Ex / Gd / TA / Fa / Po).

- Nhà bếp là 1 trong những yếu tố quyết định nhất khi mua nhà ở Mỹ
- Tương quan với giá cao (r ≈ 0.66) — gần ngang `ExterQual`

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding (cùng map với ExterQual)

---

### `Exterior1st` & `Exterior2nd` — Vật liệu bề mặt ngoài 🏷️ Nominal

Loại vật liệu bao phủ mặt ngoài ngôi nhà (`VinylSd`, `HdBoard`, `MetalSd`, `BrkFace`...).

- `Exterior2nd` chỉ điền khi nhà có 2 loại vật liệu khác nhau
- Vinyl siding (`VinylSd`) → nhà hiện đại, giá trung bình-cao
- Brick Face (`BrkFace`) → nhà kiểu cổ điển, chất lượng cao

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding

---

### `MasVnrType` & `MasVnrArea` — Tường ốp đá/gạch 🏷️ Nominal + 🔢 Numeric

- `MasVnrType`: Loại vật liệu ốp tường ngoài (BrkFace, Stone, BrkCmn, None)
- `MasVnrArea`: Diện tích ốp tường (sqft) — 0 nếu không có

Tường ốp gạch/đá → tăng giá trị thẩm mỹ → thường đi với nhà chất lượng cao.

> 🎯 **Dùng ở:** `[B4]` fill missing MasVnrType bằng "None", MasVnrArea bằng 0; `[B5]` `HasMasVnr = 1 if MasVnrArea > 0 else 0`

---

## 🏚️ Nhóm 4 — Tầng Hầm / Basement (11 cột)

> Nhà Mỹ thường có tầng hầm rộng — đây là không gian sinh sống thêm rất quan trọng. Nhóm này có nhiều cột liên quan chặt với nhau.

---

### `BsmtQual` — Chất lượng tầng hầm 📊 Ordinal

Đánh giá **chiều cao** (!) của tầng hầm — không phải chất lượng chung chung.

| Giá trị | Điểm | Chiều cao tầng hầm |
|---|---|---|
| `Ex` | 5 | ≥ 100 inch (≥ 2.54m) — rất cao, như phòng thực sự |
| `Gd` | 4 | 90–99 inch (2.28–2.51m) |
| `TA` | 3 | 80–89 inch (2.03–2.26m) — điển hình |
| `Fa` | 2 | 70–79 inch (1.78–2.00m) — hơi thấp |
| `Po` | 1 | < 70 inch — rất thấp, không dùng được nhiều |
| `NA` | 0 | **Không có tầng hầm** |

> ⚠️ `NA` ở đây = "No Basement" (không phải missing!) → điền `0` hoặc `"None"` tùy encoding

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding với map `{Ex:5, Gd:4, TA:3, Fa:2, Po:1, NA:0}`

---

### `BsmtCond` — Tình trạng tầng hầm 📊 Ordinal

Tình trạng chung của tầng hầm (độ ẩm, nứt vỡ...). Cùng thang điểm Ex/Gd/TA/Fa/Po/NA.

- Phần lớn là `TA` (độ ẩm nhẹ — bình thường)
- Tương quan với giá thấp hơn `BsmtQual`

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding

---

### `BsmtExposure` — Mức độ tiếp xúc ánh sáng tầng hầm 📊 Ordinal

Tầng hầm có cửa sổ hay lối ra vườn không?

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `Gd` | 4 | Tiếp xúc tốt (cửa ra vườn, nhiều cửa sổ) |
| `Av` | 3 | Trung bình (split level hoặc foyer) |
| `Mn` | 2 | Tối thiểu |
| `No` | 1 | Không tiếp xúc (tầng hầm kín hoàn toàn) |
| `NA` | 0 | Không có tầng hầm |

- Tầng hầm có ánh sáng → có thể dùng làm phòng sinh sống → tăng giá

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding

---

### `BsmtFinType1` & `BsmtFinType2` — Mức hoàn thiện tầng hầm 📊 Ordinal

Mức độ hoàn thiện (finish) của phần diện tích tầng hầm.

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `GLQ` | 6 | Good Living Quarters — phòng sinh sống chất lượng cao |
| `ALQ` | 5 | Average Living Quarters |
| `BLQ` | 4 | Below Average Living Quarters |
| `Rec` | 3 | Rec Room (phòng giải trí) |
| `LwQ` | 2 | Low Quality |
| `Unf` | 1 | Chưa hoàn thiện (unfinished) |
| `NA` | 0 | Không có tầng hầm |

- `BsmtFinType1` → diện tích hoàn thiện chính (`BsmtFinSF1`)
- `BsmtFinType2` → diện tích hoàn thiện thứ 2 (`BsmtFinSF2`) — nếu tầng hầm có 2 phần khác nhau

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding

---

### `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, `TotalBsmtSF` — Diện tích tầng hầm 🔢 Numeric

| Cột | Nghĩa |
|---|---|
| `BsmtFinSF1` | Diện tích phần hoàn thiện loại 1 (sqft) |
| `BsmtFinSF2` | Diện tích phần hoàn thiện loại 2 (sqft) |
| `BsmtUnfSF` | Diện tích chưa hoàn thiện (sqft) |
| `TotalBsmtSF` | **Tổng diện tích tầng hầm** = SF1 + SF2 + UnfSF |

> ✅ Kiểm tra: `BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF = TotalBsmtSF` (luôn đúng trong sạch dữ liệu)

- `TotalBsmtSF` là feature quan trọng nhất trong nhóm này (r ≈ 0.61 với SalePrice)
- Nhà không có tầng hầm → tất cả = 0

> 🎯 **Dùng ở:** `[B4]` fill missing bằng 0 (nhà không có tầng hầm), `[B5]` tạo `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`

---

### `BsmtFullBath` & `BsmtHalfBath` — Phòng tắm tầng hầm 🔢 Numeric

Số phòng tắm đầy đủ (full bath) và phòng tắm nhỏ (half bath — chỉ có bồn cầu + lavabo) trong tầng hầm.

- Nhà không có tầng hầm → = 0
- `FullBath > HalfBath` về giá trị (full bath tiện lợi hơn)

> 🎯 **Dùng ở:** `[B5]` có thể gộp: `TotalBath = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath`

---

### `Heating` & `HeatingQC` — Hệ thống sưởi 🏷️ Nominal + 📊 Ordinal

- `Heating`: Loại hệ thống sưởi (`GasA` chiếm ~98% — lò sưởi gas ép — tiêu chuẩn Mỹ)
- `HeatingQC`: Chất lượng hệ thống sưởi (Ex/Gd/TA/Fa/Po)

> 🎯 **Dùng ở:** `Heating` → `[B4]` One-Hot (nhưng low-variance); `HeatingQC` → `[B4]` Ordinal Encoding

---

### `CentralAir` — Máy lạnh trung tâm 🏷️ Nominal → Binary

Có điều hòa không khí trung tâm không? `Y` (có) hoặc `N` (không).

- Gần 94% nhà có máy lạnh trung tâm
- Ảnh hưởng vừa phải đến giá (r ≈ 0.25)
- Có thể encode thẳng thành 0/1

> 🎯 **Dùng ở:** `[B4]` Binary Encoding (Y=1, N=0) — không cần One-Hot vì chỉ 2 giá trị

---

### `Electrical` — Hệ thống điện 🏷️ Nominal

Loại hệ thống điện: `SBrkr` (cầu dao tự động — hiện đại, chiếm ~91%), `FuseA/F/P` (hộp cầu chì — cũ), `Mix`.

- Nhà cũ thường có `FuseA/F/P` → ảnh hưởng đến giá và bảo hiểm
- Missing 1 giá trị → điền `SBrkr` (mode)

> 🎯 **Dùng ở:** `[B4]` fill mode, One-Hot Encoding

---

## 📐 Nhóm 5 — Diện Tích Sàn & Phòng Bên Trong (14 cột)

> Nhóm này ghi lại diện tích từng tầng, số phòng và tiện nghi bên trong nhà. Diện tích = yếu tố quyết định giá hàng đầu.

---

### `1stFlrSF` — Diện tích tầng 1 🔢 Numeric ⭐⭐

Diện tích tầng 1 (square feet). Với nhà 1 tầng = toàn bộ diện tích sống.

- Tương quan cao với giá (r ≈ 0.60)
- Luôn > 0 (mọi nhà đều có tầng 1)
- Không có missing values

> 🎯 **Dùng ở:** `[B5]` → `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`

---

### `2ndFlrSF` — Diện tích tầng 2 🔢 Numeric

Diện tích tầng 2. = 0 nếu là nhà 1 tầng.

- Khoảng 57% nhà trong dataset có `2ndFlrSF = 0` (nhà 1 tầng)
- Liên quan chặt với `HouseStyle`: nhà `1Story` → `2ndFlrSF = 0`

> 🎯 **Dùng ở:** `[B5]` → `Has2ndFloor = 1 if 2ndFlrSF > 0 else 0`; gộp vào `TotalSF`

---

### `LowQualFinSF` — Diện tích hoàn thiện kém 🔢 Numeric

Diện tích sàn có chất lượng hoàn thiện thấp (tất cả các tầng gộp lại).

- Gần 95% nhà có giá trị = 0 → **low-variance**, ít ảnh hưởng
- Thường bị bỏ qua hoặc giữ lại không đáng kể

> 🎯 **Dùng ở:** `[EDA]` kiểm tra, thường ít được dùng trong feature engineering

---

### `GrLivArea` — Diện tích sinh sống trên mặt đất 🔢 Numeric ⭐⭐⭐

**Tổng diện tích sống "above grade"** (trên mặt đất, không tính tầng hầm).

`GrLivArea = 1stFlrSF + 2ndFlrSF + LowQualFinSF`

- **Feature quan trọng thứ 2** sau `OverallQual` (r ≈ 0.71 với SalePrice)
- Là cột dễ thấy outlier nhất: 2 nhà diện tích > 4000 sqft nhưng bán giá rất thấp (đã xóa ở Buổi 4!)
- Phân phối hơi lệch phải → có thể log transform

> 🎯 **Dùng ở:** `[EDA]` scatter plot với SalePrice, `[B4]` xóa outlier, `[B5]` tạo interaction feature `GrLivArea * OverallQual`

---

### `FullBath` & `HalfBath` — Phòng tắm tầng trên 🔢 Numeric

Số phòng tắm đầy đủ và phòng tắm nhỏ **trên mặt đất** (không tính tầng hầm).

- `FullBath`: có bồn tắm/vòi hoa sen + bồn cầu + lavabo (đầy đủ)
- `HalfBath`: chỉ có bồn cầu + lavabo (nửa phòng tắm)
- Hầu hết nhà có 1–2 FullBath

> 🎯 **Dùng ở:** `[B5]` gộp thành `TotalBath = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath`

---

### `BedroomAbvGr` — Số phòng ngủ 🔢 Numeric

Số phòng ngủ **trên mặt đất** (không tính tầng hầm).

- Nhà trung bình có 3 phòng ngủ
- Tương quan trung bình với giá (r ≈ 0.17) — vì nhà nhỏ có thể có nhiều phòng ngủ nhỏ

> 🎯 **Dùng ở:** `[B5]` có thể tạo "diện tích trung bình mỗi phòng ngủ = GrLivArea / BedroomAbvGr"

---

### `KitchenAbvGr` — Số bếp 🔢 Numeric

Số bếp trên mặt đất. Gần 100% nhà có đúng 1 bếp → **gần như vô dụng** (no variance).

> 🎯 **Dùng ở:** thường bị **drop** hoặc giữ như feature bình thường

---

### `TotRmsAbvGrd` — Tổng số phòng 🔢 Numeric

Tổng số phòng trên mặt đất (không tính phòng tắm).

- Gồm: phòng ngủ + phòng khách + phòng ăn + phòng làm việc...
- Tương quan vừa phải với giá (r ≈ 0.53)
- Liên quan chặt với `GrLivArea` (nhà nhiều phòng = nhà to)

> 🎯 **Dùng ở:** `[B5]` tạo "diện tích trung bình mỗi phòng = GrLivArea / TotRmsAbvGrd"

---

### `Functional` — Mức độ hoạt động bình thường 📊 Ordinal

Ngôi nhà có hoạt động đầy đủ chức năng không? Hay bị hư hỏng gì?

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `Typ` | 7 | Bình thường, không có vấn đề (chiếm ~93%) |
| `Min1` | 6 | Khấu trừ nhỏ loại 1 |
| `Min2` | 5 | Khấu trừ nhỏ loại 2 |
| `Mod` | 4 | Khấu trừ vừa |
| `Maj1` | 3 | Khấu trừ lớn loại 1 |
| `Maj2` | 2 | Khấu trừ lớn loại 2 |
| `Sev` | 1 | Hư hỏng nghiêm trọng |
| `Sal` | 0 | Chỉ cứu được vật liệu (salvage) |

- Missing 2 giá trị → điền `Typ` (mode)

> 🎯 **Dùng ở:** `[B4]` fill mode + Ordinal Encoding

---

### `Fireplaces` & `FireplaceQu` — Lò sưởi 🔢 + 📊 Ordinal

- `Fireplaces`: Số lò sưởi trong nhà (0, 1, 2, 3)
- `FireplaceQu`: Chất lượng lò sưởi (Ex/Gd/TA/Fa/Po/NA)

Lò sưởi = yếu tố sang trọng tại Mỹ → tăng giá đáng kể.

- `NA` trong `FireplaceQu` = không có lò sưởi (tương ứng `Fireplaces = 0`)

> 🎯 **Dùng ở:** `[B4]` FireplaceQu: Ordinal Encoding (NA=0); `[B5]` `HasFireplace = 1 if Fireplaces > 0 else 0`

---

### `PoolArea` & `PoolQC` — Bể bơi 🔢 + 📊 Ordinal

- `PoolArea`: Diện tích bể bơi (sqft) — 0 nếu không có
- `PoolQC`: Chất lượng bể bơi (Ex/Gd/TA/Fa/NA)

Chỉ ~0.5% nhà có bể bơi → cực kỳ hiếm trong dataset này → ít ảnh hưởng đến model tổng thể.

> 🎯 **Dùng ở:** `[B4]` fill "None" cho PoolQC; `[B5]` `HasPool = 1 if PoolArea > 0 else 0`

---

### `MiscFeature` & `MiscVal` — Tiện ích đặc biệt khác 🏷️ + 🔢

- `MiscFeature`: Thứ không được phân loại vào đâu khác (Thang máy, Sân tennis, Nhà kho phụ...)
- `MiscVal`: Giá trị ước tính của tiện ích đó ($)

Cực hiếm, ít ảnh hưởng. `NA` = không có.

> 🎯 **Dùng ở:** `[B4]` fill "None" + 0; thường ít ảnh hưởng đến model

---

### `Fence` — Hàng rào 📊 Ordinal (có thể coi Nominal)

Chất lượng hàng rào: `GdPrv` (riêng tư tốt), `MnPrv` (riêng tư tối thiểu), `GdWo` (gỗ tốt), `MnWw` (gỗ tối thiểu), `NA` (không có hàng rào).

- Missing ~80% → NA = không có hàng rào
- Ảnh hưởng trung bình đến giá

> 🎯 **Dùng ở:** `[B4]` fill "None", One-Hot hoặc Ordinal Encoding

---

## 🚗 Nhóm 6 — Garage (9 cột)

> Garage = yếu tố quan trọng ở Mỹ — hầu hết người Mỹ đều có xe và cần garage. Nhóm này có nhiều missing tương tự tầng hầm.

---

### `GarageType` — Loại garage 🏷️ Nominal

Vị trí/kiểu garage so với ngôi nhà.

| Giá trị | Nghĩa | Giá trị ảnh hưởng |
|---|---|---|
| `Attchd` | Gắn liền với nhà (phổ biến nhất ~59%) | Cao ↑ |
| `Detchd` | Tách rời khỏi nhà (~27%) | Trung bình |
| `BuiltIn` | Tích hợp trong nhà (~6%) | Cao ↑ |
| `Basment` | Garage trong tầng hầm | Trung bình |
| `CarPort` | Mái che hở (carport) | Thấp ↓ |
| `2Types` | Có 2 loại garage | Hiếm |
| `NA` | Không có garage | Thấp ↓ |

> 🎯 **Dùng ở:** `[B4]` fill "None" cho missing, One-Hot Encoding

---

### `GarageYrBlt` — Năm xây garage 🔢 Numeric

Năm garage được xây dựng. Thường = `YearBuilt` (xây cùng lúc với nhà).

- Missing ~5.5% → nhà không có garage → điền `0` hoặc median
- Có outlier năm 2207 (lỗi nhập liệu) → xử lý ở bước cleaning

> 🎯 **Dùng ở:** `[B4]` fill 0 hoặc YearBuilt cho missing; `[B5]` tạo `GarageAge = YrSold - GarageYrBlt`

---

### `GarageFinish` — Mức độ hoàn thiện garage 📊 Ordinal

Nội thất bên trong garage được hoàn thiện đến mức nào.

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `Fin` | 3 | Hoàn thiện đầy đủ (sàn sơn, tường ốp...) |
| `RFn` | 2 | Hoàn thiện thô (Rough Finished) |
| `Unf` | 1 | Chưa hoàn thiện (bê tông thô) |
| `NA` | 0 | Không có garage |

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding

---

### `GarageCars` — Sức chứa garage (số xe) 🔢 Numeric ⭐⭐

Garage chứa được bao nhiêu xe.

- Tương quan cao với giá (r ≈ 0.64) — thường cao hơn `GarageArea`!
- Phần lớn nhà có `GarageCars = 2`
- Missing 1 giá trị → fill = 0 (không có garage)

> 🎯 **Dùng ở:** `[B4]` fill 0, `[B6]` feature quan trọng

---

### `GarageArea` — Diện tích garage 🔢 Numeric ⭐

Diện tích garage (sqft). Liên quan chặt với `GarageCars`.

- Tương quan cao với giá (r ≈ 0.62)
- Nên dùng `GarageCars` hoặc `GarageArea`, không cần cả 2 (multicollinearity)
- Missing 1 → fill = 0

> 🎯 **Dùng ở:** `[B4]` fill 0; `[B5]` `HasGarage = 1 if GarageArea > 0 else 0`

---

### `GarageQual` & `GarageCond` — Chất lượng & Tình trạng garage 📊 Ordinal

- `GarageQual`: Chất lượng garage (Ex/Gd/TA/Fa/Po/NA)
- `GarageCond`: Tình trạng hiện tại của garage (Ex/Gd/TA/Fa/Po/NA)

Cả hai đều có 95%+ giá trị là `TA` → **rất ít variance** → ít đóng góp vào model.

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding, nhưng contribution thấp

---

### `PavedDrive` — Đường vào nhà 📊 Ordinal

Lối vào nhà (từ đường đến cửa garage) có được lát không?

| Giá trị | Điểm | Nghĩa |
|---|---|---|
| `Y` | 2 | Được lát nhựa đường (Paved) |
| `P` | 1 | Lát một phần |
| `N` | 0 | Đất/đá dăm |

> 🎯 **Dùng ở:** `[B4]` Ordinal Encoding (Y=2, P=1, N=0)

---

## 🌿 Nhóm 6b — Hiên & Diện Tích Ngoài Trời (5 cột)

---

### `WoodDeckSF` — Sàn gỗ ngoài trời 🔢 Numeric

Diện tích sàn gỗ (deck) ngoài trời (sqft). = 0 nếu không có.

> 🎯 **Dùng ở:** `[B5]` gộp vào `TotalPorch = WoodDeckSF + OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch`

---

### `OpenPorchSF`, `EnclosedPorch`, `3SsnPorch`, `ScreenPorch` — Các loại hiên 🔢 Numeric

| Cột | Nghĩa |
|---|---|
| `OpenPorchSF` | Hiên mở (open porch) — phổ biến nhất |
| `EnclosedPorch` | Hiên có tường bao kín |
| `3SsnPorch` | Hiên 3 mùa (có cửa sổ, dùng được 3 mùa) |
| `ScreenPorch` | Hiên có lưới chống muỗi |

- Tất cả = 0 nếu không có
- Ít ảnh hưởng riêng lẻ, nhưng tổng `TotalPorch` có ý nghĩa hơn

> 🎯 **Dùng ở:** `[B5]` → `TotalPorch` (tổng tất cả diện tích hiên), `HasPorch = 1 if TotalPorch > 0 else 0`

---

## 💰 Nhóm 7 — Thông Tin Giao Dịch Bán (6 cột)

> Nhóm này mô tả **bối cảnh của giao dịch mua bán** — khi nào bán, bán kiểu gì. Cần cẩn thận vì một số cột này **không nên dùng để predict** (data leakage tiềm năng).

---

### `MoSold` — Tháng bán 🔢 Numeric → có thể Nominal

Tháng giao dịch được thực hiện (1–12).

- Thị trường bất động sản có tính mùa vụ: mùa hè (5–8) thường bán được giá cao hơn
- Có thể encode theo mùa: `Season = Spring/Summer/Fall/Winter`

> 🎯 **Dùng ở:** `[B5]` tạo feature `Season` hoặc `IsHighSeason = 1 if MoSold in [5,6,7,8] else 0`

---

### `YrSold` — Năm bán 🔢 Numeric

Năm giao dịch (2006–2010).

- Dataset trải qua giai đoạn khủng hoảng nhà đất Mỹ 2008 → 2009 giá có xu hướng thấp hơn
- **Cực kỳ quan trọng** để tính `HouseAge = YrSold - YearBuilt`

> 🎯 **Dùng ở:** `[B5]` tính `HouseAge`, `RemodAge`, `GarageAge` — sau đó có thể drop `YrSold` khỏi features

---

### `SaleType` — Loại hình bán 🏷️ Nominal

Hình thức pháp lý của giao dịch mua bán.

| Giá trị | Nghĩa | Ghi chú |
|---|---|---|
| `WD` | Warranty Deed thông thường (~82%) | Mua bán bình thường |
| `New` | Nhà mới xây, bán lần đầu | Giá thường cao |
| `COD` | Bán qua tòa án (tịch thu tài sản, thừa kế) | Giá thường thấp |
| `Con*` | Bán theo hợp đồng (trả góp...) | Đặc biệt |
| `CWD` | Warranty Deed trả tiền mặt | |
| `Oth` | Khác | |

> 🎯 **Dùng ở:** `[B4]` One-Hot Encoding; `[B5]` `IsNewHome = 1 if SaleType == "New" else 0`

---

### `SaleCondition` — Điều kiện bán 🏷️ Nominal ⚠️

Điều kiện/hoàn cảnh của giao dịch.

| Giá trị | Nghĩa | Ảnh hưởng |
|---|---|---|
| `Normal` | Bán bình thường (~82%) | Giá thị trường |
| `Abnorml` | Bán bất thường (tịch biên, thừa kế...) | Giá thấp hơn ↓ |
| `Partial` | Nhà chưa hoàn thiện khi đánh giá | Giá biến động |
| `Family` | Bán trong gia đình | Giá thấp hơn ↓ |
| `Alloca` | Phân bổ — bán kèm nhà + garage riêng | Đặc biệt |
| `AdjLand` | Mua thêm đất kề bên | Hiếm |

> ⚠️ `Abnorml` và `Family` → giá thấp bất thường không phản ánh giá trị thực → cần lưu ý khi EDA

> 🎯 **Dùng ở:** `[EDA]` loại outlier price từ Abnorml/Family; `[B4]` One-Hot Encoding

---

### `Id` — Định danh dòng 🔢 Numeric ❌ Không dùng!

Số thứ tự định danh từng căn nhà trong dataset.

- **Không có ý nghĩa thống kê** — chỉ là số thứ tự
- Phải **xóa** trước khi training model
- Giữ lại để map kết quả khi submit Kaggle

> 🎯 **Dùng ở:** `[B4]` drop trước khi training; giữ `test_id` để tạo file submission

---

### `SalePrice` — Giá bán 🎯 TARGET ⭐⭐⭐

Giá bán cuối cùng của ngôi nhà (USD). Đây là **biến mục tiêu** cần dự đoán.

- Dao động từ ~$34,900 đến ~$755,000
- Trung bình ~$180,921
- Phân phối **lệch phải** (right-skewed) → cần `log1p()` transform
- Sau log transform → phân phối gần chuẩn → RMSE đo trên log scale → RMSLE

```python
import numpy as np
y = np.log1p(train_df['SalePrice'])   # transform
# Sau khi predict: np.expm1(y_pred)    # reverse transform
```

> 🎯 **Dùng ở:** `[EDA]` histogram + Q-Q plot, `[B4]` log transform, `[B6]` target variable, `[B7]` reverse transform để submit

---

## 📊 Tổng Kết — Bảng Tham Chiếu Nhanh

| Nhóm | Encoding | Cột tiêu biểu | Ghi chú |
|---|---|---|---|
| Vị trí & Lô đất | One-Hot | `Neighborhood`, `MSZoning` | Neighborhood quan trọng nhất |
| Loại nhà | One-Hot | `BldgType`, `HouseStyle` | `MSSubClass` dễ nhầm = nominal! |
| Chất lượng | Ordinal / Numeric trực tiếp | `OverallQual`, `ExterQual`, `KitchenQual` | Feature quan trọng nhất |
| Thời gian | Tính feature mới | `YearBuilt`, `YearRemodAdd` | → `HouseAge`, `RemodAge` |
| Tầng hầm | Ordinal + Numeric | `BsmtQual`, `TotalBsmtSF` | NA = không có tầng hầm |
| Diện tích | Numeric trực tiếp | `GrLivArea`, `1stFlrSF` | Feature quan trọng thứ 2 |
| Phòng | Numeric + Tạo mới | `FullBath`, `Fireplaces` | → `TotalBath`, `HasFireplace` |
| Garage | Ordinal + Numeric | `GarageCars`, `GarageArea` | NA = không có garage |
| Hiên | Numeric → Gộp | `OpenPorchSF`, `WoodDeckSF` | → `TotalPorch` |
| Giao dịch | One-Hot | `SaleType`, `SaleCondition` | Cẩn thận data leakage |

---

> 💡 **Mẹo học:** Khi đọc code của các Kaggle notebooks, bạn sẽ thường thấy các feature engineering từ đây được tạo ra. Đọc bảng trên giúp bạn hiểu tại sao người ta làm vậy thay vì chỉ copy code.